# ปฏิบัติการที่ 4 การค้นหาแบบมีข้อมูลและ A*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NKSR22/Introduction_to_AI_Labs/blob/main/lab04-informed-search.ipynb)

ปฏิบัติการประกอบหนังสือ *ปัญญาประดิษฐ์เบื้องต้น* (นครินทร์ ศรีปัญญา) บทที่ 2 หัวข้อ 2.3 การค้นหาแบบมีข้อมูล

## วัตถุประสงค์

เมื่อทำปฏิบัติการนี้จบ ผู้เรียนสามารถ

1. คำนวณฮิวริสติกระยะเส้นตรงและระยะแมนฮัตตัน และตรวจได้ว่าฮิวริสติกยอมรับได้ (Admissible) และคงเส้นคงวา (Consistent) หรือไม่
2. เขียน Greedy Best-First Search และ A* จาก BEST-FIRST-SEARCH ตัวเดิม แล้วตรวจผลกับตารางไล่อัลกอริทึมในหนังสือได้
3. ใช้ A* หาเส้นทาง AGV บนแผนที่ตารางของคลังสินค้า และเปรียบเทียบจำนวนช่องที่ขยายกับ BFS, UCS และ Greedy ด้วยภาพได้
4. เลือกฮิวริสติกให้ตรงกับวิธีเคลื่อนที่ (4 ทิศ หรือ 8 ทิศ) และอธิบายผลของฮิวริสติกที่ประเมินเกินจริงได้
5. เปรียบเทียบฮิวริสติก $h_1$ และ $h_2$ ของ 8-Puzzle ด้วยจำนวนโหนดและตัวประกอบการแตกกิ่งยังผลได้

## เนื้อหาในหนังสือที่เกี่ยวข้อง

- 2.3.1 ฟังก์ชันฮิวริสติก — ตัวอย่างที่ 2.12 และ 2.13
- 2.3.2 Greedy Best-First Search — ตัวอย่างที่ 2.14
- 2.3.3 A* Search — ตัวอย่างที่ 2.15, 2.16 และตารางที่ 2.9
- 2.3.4 การออกแบบฮิวริสติก — ตัวอย่างที่ 2.17 และ 2.18

## ความรู้พื้นฐานที่ควรมี

- ทำปฏิบัติการที่ 3 มาก่อน (ปฏิบัติการนี้ใช้คลาส `RouteProblem`, `Node` และฟังก์ชัน `best_first_search` ชุดเดิม ซึ่งรวมไว้ในเซลล์ "โค้ดพื้นฐาน" แล้ว)
- อ่านหัวข้อ 2.3 ของหนังสือมาก่อน

## วิธีใช้ notebook นี้

- กดปุ่ม **Open in Colab** ด้านบนเพื่อเปิดใน Google Colab แล้วรันทีละเซลล์จากบนลงล่างด้วย `Shift + Enter`
- ถ้ารันเซลล์ข้ามลำดับแล้วเกิดข้อผิดพลาด ให้เลือกเมนู **Runtime → Run all** เพื่อรันใหม่ทั้งหมด
- ส่วนที่ 6 (8-Puzzle) ใช้เวลารันรวมประมาณ 10–20 วินาที
- เซลล์ที่มี `# TODO` ในส่วน "แบบฝึกปฏิบัติ" คือส่วนที่ผู้เรียนต้องเติมโค้ดเอง

| ส่วน | หัวข้อ | ตัวอย่างในหนังสือ |
|:---:|---|---|
| 1 | ฮิวริสติกระยะเส้นตรงของผังโรงงาน | 2.12, 2.13 |
| 2 | Greedy Best-First Search | 2.14 |
| 3 | A* Search | 2.15, ตารางที่ 2.9 |
| 4 | ฮิวริสติกที่ยอมรับได้และคงเส้นคงวา | 2.16 |
| 5 | A* บนแผนที่ตารางของคลังสินค้า เทียบกับ BFS/UCS | 2.13, 2.18 |
| 6 | 8-Puzzle: เปรียบเทียบ $h_1$ กับ $h_2$ | 2.17 |
| 7 | แบบฝึกปฏิบัติ: ระยะอ็อกไทล์, Weighted A*, ความแออัดในต้นทุน | — |

## เตรียมเครื่องมือ

ปฏิบัติการนี้ใช้เฉพาะไลบรารีที่มีใน Google Colab อยู่แล้ว (`networkx`, `matplotlib`, `pandas`, `numpy` และไลบรารีมาตรฐานของ Python) จึงไม่ต้องติดตั้งเพิ่ม

> **หมายเหตุ** ป้ายกำกับบนรูปใช้ภาษาอังกฤษ เพราะ Colab ไม่มีฟอนต์ภาษาไทยติดตั้งไว้

In [ ]:
# ไม่ต้องติดตั้งไลบรารีเพิ่ม (ถ้ารันบนเครื่องตนเองแล้วขาดไลบรารี ให้รัน: !pip install -q networkx pandas matplotlib)
import heapq
import itertools
import math
import random
from collections import Counter, deque
from dataclasses import dataclass, field
from typing import Any, Optional

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

RANDOM_SEED = 42                      # กำหนด seed ให้ผลการสุ่มเหมือนเดิมทุกครั้ง
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_colwidth", None)
plt.rcParams["figure.dpi"] = 100
COLORS = {"blue": "#1F4E79", "orange": "#E07B39", "green": "#3A7D44", "gray": "#6B6B6B"}

print("networkx", nx.__version__, "| pandas", pd.__version__)

### โค้ดพื้นฐานจากปฏิบัติการที่ 3

เซลล์ต่อไปนี้รวมโค้ดที่เขียนและอธิบายไว้แล้วในปฏิบัติการที่ 3 ได้แก่ ผังโรงงาน (`FACTORY_MAP`, `COORDS`) ฟังก์ชันวาดผัง ปัญหาและโหนด (`RouteProblem`, `Node`, `expand`, `SearchResult`) BFS, BEST-FIRST-SEARCH กับ UCS และฟังก์ชันวาดแผนที่ตาราง รันเซลล์นี้ครั้งเดียวแล้วไปต่อได้เลย ถ้าต้องการทบทวนรายละเอียด ให้ดูคำอธิบายในปฏิบัติการที่ 3

ประเด็นที่ต้องจำจาก `best_first_search`: นำโหนดที่ $f$ ต่ำสุดออกก่อน ถ้า $f$ เท่ากันเลือกตามลำดับของสถานะ (ตัวอักษร) ทดสอบเป้าหมายตอนนำออก และให้เส้นทางที่ถูกกว่าแทนที่เส้นทางเดิม

In [ ]:
# ---------- โค้ดพื้นฐานจากปฏิบัติการที่ 3 (ไม่ต้องแก้) ----------
# ผังโรงงาน (ตารางที่ 2.2): จุด -> {จุดข้างเคียง: ระยะทาง (m)} เรียงจุดข้างเคียงตามตัวอักษร
FACTORY_MAP = {
    "S": {"A": 20, "C": 25, "X": 20},
    "A": {"B": 20, "S": 20},
    "B": {"A": 20, "X": 20},
    "C": {"S": 25, "Y": 30},
    "X": {"B": 20, "G": 40, "S": 20, "Y": 15},
    "Y": {"C": 30, "G": 10, "X": 15},
    "G": {"X": 40, "Y": 10},
}

# พิกัดของแต่ละจุด (m) ตามตัวอย่างที่ 2.12 ใช้วาดผังและคำนวณฮิวริสติก
COORDS = {"S": (5, 27), "A": (4, 42), "B": (19, 43), "C": (5, 3),
          "X": (24, 27), "Y": (30, 15), "G": (40, 15)}

START, GOAL = "S", "G"

def draw_factory_map(graph=None, expanded=(), path=(), title="", ax=None, node_text=None):
    """วาดผังโรงงาน: วงกลมทึบสีน้ำเงิน = จุดที่ถูกขยาย, เส้นหนาสีส้ม = เส้นทางที่ได้
    node_text: dict ข้อความเล็กใต้แต่ละจุด เช่น ค่าฮิวริสติก"""
    graph = FACTORY_MAP if graph is None else graph
    G = nx.Graph()
    G.add_nodes_from(graph)
    for u, nbrs in graph.items():
        for v, w in nbrs.items():
            G.add_edge(u, v, weight=w)
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 5))

    on_path = {frozenset(e) for e in zip(path, path[1:])}
    edges = list(G.edges())
    nx.draw_networkx_edges(
        G, COORDS, ax=ax, edgelist=edges,
        edge_color=[COLORS["orange"] if frozenset(e) in on_path else COLORS["gray"] for e in edges],
        width=[4 if frozenset(e) in on_path else 1.5 for e in edges])
    nx.draw_networkx_edge_labels(G, COORDS, ax=ax, edge_labels=nx.get_edge_attributes(G, "weight"),
                                 font_size=8, font_color=COLORS["gray"])
    nodes = list(G.nodes())
    nx.draw_networkx_nodes(G, COORDS, ax=ax, nodelist=nodes, node_size=650,
                           node_color=[COLORS["blue"] if n in expanded else "white" for n in nodes],
                           edgecolors=COLORS["blue"], linewidths=2)
    for n in nodes:
        x, y = COORDS[n]
        ax.text(x, y, n, ha="center", va="center", fontsize=12, fontweight="bold",
                color="white" if n in expanded else COLORS["blue"])
        if node_text and n in node_text:
            ax.text(x, y - 3.5, f"({node_text[n]})", ha="center", va="top", fontsize=9, color=COLORS["gray"])
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.margins(0.15)
    ax.axis("off")
    return ax

class RouteProblem:
    """ปัญหาหาเส้นทางบนกราฟ: สถานะ = ชื่อจุด, การกระทำ = วิ่งไปจุดข้างเคียง, ต้นทุน = ระยะทาง"""

    def __init__(self, graph, initial, goal):
        self.graph = graph
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        return sorted(self.graph[state])            # ข้อตกลง: สร้างลูกตามลำดับตัวอักษร

    def result(self, state, action):
        return action                                # การกระทำ "ไป X" ทำให้อยู่ที่ X

    def action_cost(self, state, action, next_state):
        return self.graph[state][next_state]

    def is_goal(self, state):
        return state == self.goal


@dataclass
class Node:
    """โหนดในต้นไม้ค้นหา (หัวข้อ 2.2.1) เก็บสถานะ โหนดแม่ การกระทำ ต้นทุนเส้นทาง g และความลึก"""
    state: Any
    parent: Optional["Node"] = field(default=None, repr=False)
    action: Any = None
    path_cost: float = 0
    depth: int = 0


def expand(problem, node):
    """สร้างโหนดลูกทีละโหนดตามรหัสเทียม EXPAND"""
    for action in problem.actions(node.state):
        next_state = problem.result(node.state, action)
        cost = node.path_cost + problem.action_cost(node.state, action, next_state)
        yield Node(next_state, node, action, cost, node.depth + 1)


def path_states(node):
    """ย้อนตาม parent จากโหนดเป้าหมายกลับไปถึงราก แล้วกลับลำดับ"""
    states = []
    while node is not None:
        states.append(node.state)
        node = node.parent
    return states[::-1]


@dataclass
class SearchResult:
    """ผลการค้นหา: โหนดคำตอบ (หรือ None) และสถิติที่ใช้เปรียบเทียบอัลกอริทึม"""
    algorithm: str
    node: Optional[Node]
    expanded_nodes: list      # โหนดที่ถูกขยายตามลำดับ
    tested: list              # สถานะที่ถูกทดสอบเป้าหมายตามลำดับ
    generated: list           # โหนดที่ถูกสร้างและเก็บไว้ (รวมราก)
    status: str               # "found", "failure" หรือ "cutoff"

    @property
    def expanded(self):
        return [n.state for n in self.expanded_nodes]

    @property
    def path(self):
        return path_states(self.node) if self.node else []

    @property
    def cost(self):
        return self.node.path_cost if self.node else None

    def summary(self, show_path=True):
        row = {"อัลกอริทึม": self.algorithm}
        if show_path:
            row["เส้นทางที่ได้"] = " → ".join(map(str, self.path)) or "-"
        row["ต้นทุน"] = num(self.cost)
        row["จำนวนการกระทำ"] = len(self.path) - 1 if self.path else None
        row["จำนวนโหนดที่ขยาย"] = len(self.expanded_nodes)
        return row


# ---------- ตัวช่วยแสดงผล ----------
def num(x):
    """แสดงจำนวนเต็มเป็นจำนวนเต็ม และทศนิยมไม่เกิน 2 ตำแหน่ง"""
    if x is None:
        return None
    return int(x) if float(x).is_integer() else round(float(x), 2)


def fmt_states(items, brackets="[]"):
    return brackets[0] + ", ".join(str(s) for s in items) + brackets[1]


def log(trace, row):
    """บันทึกหนึ่งแถวของตารางไล่อัลกอริทึม (ถ้าผู้เรียกส่งรายการ trace มา)"""
    if trace is not None:
        trace.append(row)


def show_trace(trace):
    """แสดงตารางไล่อัลกอริทึมโดยใช้คอลัมน์แรกเป็นดัชนี"""
    table = pd.DataFrame(trace)
    display(table.set_index(table.columns[0]))

def breadth_first_search(problem, trace=None):
    """BFS แบบกราฟ: frontier เป็นคิว FIFO, reached เป็นเซตของสถานะ
    ทดสอบเป้าหมายทันทีที่สร้างโหนดลูก (Early Goal Test)"""
    node = Node(problem.initial)
    expanded, tested, generated = [], [node.state], [node]
    if problem.is_goal(node.state):
        return SearchResult("BFS", node, expanded, tested, generated, "found")
    frontier = deque([node])
    reached = {problem.initial: True}       # dict ใช้แทนเซตที่จำลำดับการเพิ่มได้

    def row(step, expanded_state, notes, done=False):
        if trace is None:                           # ไม่ต้องสร้างข้อความถ้าไม่ได้ขอตาราง
            return None
        return {"ขั้น": step, "โหนดที่ขยาย": expanded_state, "ลูกที่สร้าง (ผลการตรวจ)": notes,
                "frontier หลังขั้นนี้": "—" if done else fmt_states(n.state for n in frontier),
                "reached หลังขั้นนี้": "—" if done else fmt_states(reached, "{}")}

    log(trace, row(0, "—", f"(ตรวจ {node.state}: ไม่ใช่เป้าหมาย)"))
    while frontier:
        node = frontier.popleft()                   # นำโหนดที่หัวคิวออก
        expanded.append(node)
        notes = []
        for child in expand(problem, node):
            tested.append(child.state)
            if problem.is_goal(child.state):        # ทดสอบตั้งแต่ตอนสร้าง
                generated.append(child)
                notes.append(f"{child.state} (เป้าหมาย!)")
                log(trace, row(len(expanded), node.state, ", ".join(notes), done=True))
                return SearchResult("BFS", child, expanded, tested, generated, "found")
            if child.state in reached:
                notes.append(f"{child.state} (พบแล้ว ข้าม)")
            else:
                reached[child.state] = True
                frontier.append(child)              # ต่อท้ายคิว
                generated.append(child)
                notes.append(f"{child.state} (ใหม่)")
        log(trace, row(len(expanded), node.state, ", ".join(notes)))
    return SearchResult("BFS", None, expanded, tested, generated, "failure")

def best_first_search(problem, f, name, describe, trace=None):
    """BEST-FIRST-SEARCH (หัวข้อ 2.2.1): frontier เป็นคิวลำดับความสำคัญเรียงตาม f(node)
    - ถ้า f เท่ากัน เลือกตามลำดับของสถานะ (ตัวอักษร) ตามข้อตกลงของหนังสือ
    - ทดสอบเป้าหมายตอน "นำออก" จาก frontier
    - ถ้าพบสถานะเดิมด้วยต้นทุน g ที่ต่ำกว่า ให้เส้นทางใหม่แทนที่เส้นทางเดิม
    describe(node) คืนข้อความของโหนดสำหรับตารางไล่อัลกอริทึม"""
    root = Node(problem.initial)
    counter = itertools.count()                 # ใช้กันกรณีค่า f และสถานะเท่ากันทุกอย่าง
    heap = [(f(root), root.state, next(counter), root)]
    in_frontier = {root.state: root}            # โหนดล่าสุดของแต่ละสถานะที่ยังรออยู่ใน frontier
    reached = {root.state: root}
    expanded, tested, generated = [], [], [root]

    def frontier_text():
        waiting = sorted(in_frontier.values(), key=lambda n: (f(n), n.state))
        return fmt_states(f"{n.state}:{num(f(n))}" for n in waiting)

    if trace is not None:
        log(trace, {"ขั้น": 0, "นำออก": "—", "เป้าหมาย?": "—", "ลูกที่สร้าง (ผล)": describe(root),
                    "frontier หลังขั้นนี้": frontier_text(), "ขยายแล้ว": "{ }"})
    while heap:
        _, _, _, node = heapq.heappop(heap)
        if in_frontier.get(node.state) is not node:
            continue                            # โหนดเก่าที่ถูกเส้นทางที่ถูกกว่าแทนที่ไปแล้ว ไม่ใช้
        del in_frontier[node.state]
        tested.append(node.state)
        if problem.is_goal(node.state):
            if trace is not None:
                log(trace, {"ขั้น": len(expanded) + 1, "นำออก": describe(node), "เป้าหมาย?": "✓",
                            "ลูกที่สร้าง (ผล)": "—", "frontier หลังขั้นนี้": "—", "ขยายแล้ว": "—"})
            return SearchResult(name, node, expanded, tested, generated, "found")
        expanded.append(node)
        notes = []
        for child in expand(problem, node):
            old = reached.get(child.state)
            if old is not None and child.path_cost >= old.path_cost:
                if trace is not None:
                    notes.append(f"{describe(child)} (เดิม g={num(old.path_cost)} ข้าม)")
                continue
            if trace is not None:
                notes.append(f"{describe(child)} (ใหม่)" if old is None
                             else f"{describe(child)} (เดิม g={num(old.path_cost)} → ปรับ)")
            reached[child.state] = child
            in_frontier[child.state] = child
            generated.append(child)
            heapq.heappush(heap, (f(child), child.state, next(counter), child))
        if trace is not None:
            log(trace, {"ขั้น": len(expanded), "นำออก": describe(node), "เป้าหมาย?": "✗",
                        "ลูกที่สร้าง (ผล)": "; ".join(notes), "frontier หลังขั้นนี้": frontier_text(),
                        "ขยายแล้ว": fmt_states((n.state for n in expanded), "{}")})
    return SearchResult(name, None, expanded, tested, generated, "failure")


def uniform_cost_search(problem, trace=None):
    """UCS = BEST-FIRST-SEARCH ที่ f(n) = g(n)"""
    return best_first_search(problem, f=lambda n: n.path_cost, name="UCS",
                             describe=lambda n: f"{n.state}:{num(n.path_cost)}", trace=trace)

def draw_grid(grid, path=(), expanded=(), title="", ax=None):
    """วาดแผนที่ตาราง: สีเทา = ชั้นวาง/ผนัง (#), สีฟ้าอ่อน = ช่องที่ถูกขยาย, เส้นสีส้ม = เส้นทางที่ได้"""
    n_rows, n_cols = len(grid), len(grid[0])
    image = np.ones((n_rows, n_cols, 3))
    for (r, c) in expanded:
        image[r, c] = (0.78, 0.87, 0.96)
    for r in range(n_rows):
        for c in range(n_cols):
            if grid[r][c] == "#":
                image[r, c] = (0.42, 0.42, 0.42)
    if ax is None:
        _, ax = plt.subplots(figsize=(0.4 * n_cols + 1, 0.4 * n_rows + 1))
    ax.imshow(image, interpolation="nearest")
    if path:
        ax.plot([c for r, c in path], [r for r, c in path], color=COLORS["orange"], linewidth=3)
    for r in range(n_rows):
        for c in range(n_cols):
            if grid[r][c] in "SG":
                ax.text(c, r, grid[r][c], ha="center", va="center", fontsize=10, fontweight="bold",
                        color="white", bbox=dict(boxstyle="circle,pad=0.2", ec="none",
                        fc=COLORS["blue"] if grid[r][c] == "S" else COLORS["green"]))
    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="#DDDDDD", linewidth=0.5)
    ax.tick_params(which="both", length=0, labelsize=7)
    ax.set_title(title, fontsize=10)
    return ax


factory_problem = RouteProblem(FACTORY_MAP, START, GOAL)
ucs_result = uniform_cost_search(factory_problem)
print("ทดสอบ UCS:", " → ".join(ucs_result.path), f"({ucs_result.cost} m) ขยาย {len(ucs_result.expanded)} โหนด")

## ส่วนที่ 1 ฮิวริสติกระยะเส้นตรงของผังโรงงาน

ฟังก์ชันฮิวริสติก $h(n)$ คือค่าประมาณต้นทุนเส้นทางที่ถูกที่สุดจากสถานะของโหนด $n$ ไปยังเป้าหมาย (หัวข้อ 2.3.1) ฮิวริสติกที่ดีต้องมี $h(\text{เป้าหมาย}) = 0$ และคำนวณได้เร็วโดยไม่ต้องค้นหา

สำหรับผังโรงงาน ใช้ **ระยะทางเส้นตรง** จากพิกัดของแต่ละจุดถึง G(40, 15)

$$h_{\text{SLD}}(n) = \sqrt{(x_G - x_n)^2 + (y_G - y_n)^2}$$

ค่าที่ได้ควรตรงกับตารางที่ 2.8 ของหนังสือ (พิกัดถูกเลือกให้ได้จำนวนเต็ม)

In [ ]:
def straight_line_distance(a, b):
    """ระยะทางเส้นตรง (ระยะยุคลิด) ระหว่างจุด a กับ b จากพิกัดใน COORDS"""
    (x1, y1), (x2, y2) = COORDS[a], COORDS[b]
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


H_SLD = {node: straight_line_distance(node, GOAL) for node in FACTORY_MAP}


def h_sld(state):
    """ฮิวริสติกที่ส่งให้อัลกอริทึม: รับสถานะ คืนค่าประมาณระยะถึง G"""
    return H_SLD[state]


goal_x, goal_y = COORDS[GOAL]
pd.DataFrame([{"จุด n": n, "Δx = 40 − x": goal_x - x, "Δy = 15 − y": goal_y - y,
               "Δx² + Δy²": (goal_x - x) ** 2 + (goal_y - y) ** 2, "h(n) (m)": num(H_SLD[n])}
              for n, (x, y) in COORDS.items()]).set_index("จุด n")

In [ ]:
draw_factory_map(node_text={n: num(h) for n, h in H_SLD.items()},
                 title="Factory map: (h) = straight-line distance to G")
plt.show()

**ตัวอย่างที่ 2.13 ระยะแมนฮัตตันกับระยะเส้นตรง** เมื่อ AGV วิ่งได้เฉพาะ 4 ทิศบนแผนที่ตาราง จาก (1, 1) ไป (4, 3) ระยะแมนฮัตตันคือ $|4-1| + |3-1| = 5$ ส่วนระยะเส้นตรงคือ $\sqrt{13} \approx 3.61$ ทั้งสองไม่เกินต้นทุนจริง แต่ระยะแมนฮัตตันใกล้ต้นทุนจริงมากกว่า (จะเห็นผลในส่วนที่ 5)

In [ ]:
def manhattan(cell_a, cell_b):
    return abs(cell_a[0] - cell_b[0]) + abs(cell_a[1] - cell_b[1])


def euclidean(cell_a, cell_b):
    return math.dist(cell_a, cell_b)


print("แมนฮัตตัน (1,1)→(4,3) =", manhattan((1, 1), (4, 3)))
print("เส้นตรง   (1,1)→(4,3) =", round(euclidean((1, 1), (4, 3)), 2))

**คำถามชวนคิด 1:** ฮิวริสติกบอกว่า X ($h = 20$) ใกล้เป้าหมายกว่า A ($h = 45$) มาก แต่ทางวิ่งจริงจาก X ไป G ตรง ๆ ยาว 40 m ฮิวริสติก "ผิด" หรือไม่ และข้อมูลใดที่ระยะเส้นตรงไม่รู้

## ส่วนที่ 2 Greedy Best-First Search

Greedy Best-First Search ขยายโหนดที่ฮิวริสติกบอกว่าใกล้เป้าหมายที่สุดก่อนเสมอ คือ BEST-FIRST-SEARCH ที่ใช้ $f(n) = h(n)$ ส่วน A* ใช้ $f(n) = g(n) + h(n)$ ทั้งสองจึงเขียนได้ในบรรทัดเดียวด้วย `best_first_search` ตัวเดิม ต่างกันเพียงฟังก์ชัน `f` ที่ส่งเข้าไป

ในตารางไล่ Greedy คอลัมน์ "นำออก" แสดง `สถานะ:h [g=...]` โดยแสดง $g$ ไว้เปรียบเทียบเท่านั้น Greedy ไม่ใช้ $g$ ในการเลือก ควรได้ตรงกับตัวอย่างที่ 2.14 คือ S → X → G (60 m) ขยายเพียง 2 โหนด

In [ ]:
def greedy_best_first_search(problem, h, trace=None, name="Greedy"):
    """Greedy Best-First Search = BEST-FIRST-SEARCH ที่ f(n) = h(n)  (h รับสถานะ คืนค่าประมาณ)"""
    return best_first_search(
        problem, f=lambda n: h(n.state), name=name, trace=trace,
        describe=lambda n: f"{n.state}:{num(h(n.state))} [g={num(n.path_cost)}]")


def astar_search(problem, h, trace=None, name="A*"):
    """A* = BEST-FIRST-SEARCH ที่ f(n) = g(n) + h(n)"""
    return best_first_search(
        problem, f=lambda n: n.path_cost + h(n.state), name=name, trace=trace,
        describe=lambda n: f"{n.state}: {num(n.path_cost)} + {num(h(n.state))} = {num(n.path_cost + h(n.state))}")


greedy_trace = []
greedy_result = greedy_best_first_search(factory_problem, h_sld, trace=greedy_trace)
show_trace(greedy_trace)
print("เส้นทาง:", " → ".join(greedy_result.path), "| ระยะทาง", greedy_result.cost, "m",
      "| ขยาย", len(greedy_result.expanded), "โหนด:", greedy_result.expanded)

**คำถามชวนคิด 2:** ที่ขั้นที่ 2 Greedy เลือก G ($h = 0$) ก่อน Y ($h = 10$) ทั้งที่ทาง X → Y → G ยาวเพียง 25 m เทียบกับ X → G 40 m ข้อมูลใดที่ Greedy ไม่ได้นำมาพิจารณา

## ส่วนที่ 3 A* Search

A* ประเมินโหนดด้วย $f(n) = g(n) + h(n)$ คือค่าประมาณต้นทุนรวมของคำตอบที่ดีที่สุดที่ผ่านโหนด $n$ ในตาราง คอลัมน์ "นำออก" และ "ลูกที่สร้าง" แสดงการคำนวณ $g + h = f$ ของทุกโหนด ควรได้ตรงกับตัวอย่างที่ 2.15 คือ S → X → Y → G (45 m) ขยายเพียง 3 โหนด (S, X, Y)

In [ ]:
astar_trace = []
astar_result = astar_search(factory_problem, h_sld, trace=astar_trace)
show_trace(astar_trace)
print("เส้นทาง:", " → ".join(astar_result.path), "| ระยะทาง", astar_result.cost, "m",
      "| ขยาย", len(astar_result.expanded), "โหนด:", astar_result.expanded)

### 3.1 เปรียบเทียบการค้นหาแบบดีที่สุดก่อน 3 แบบ (ตารางที่ 2.9)

UCS, Greedy และ A* เป็นอัลกอริทึมเดียวกันที่ต่างเพียงฟังก์ชัน $f$ ตารางและรูปด้านล่างแสดงว่า A* ได้คำตอบที่ดีที่สุดเท่ากับ UCS แต่ขยายโหนดน้อยกว่าครึ่ง

In [ ]:
best_first_results = [ucs_result, greedy_result, astar_result]
summary = pd.DataFrame([r.summary() for r in best_first_results]).set_index("อัลกอริทึม")
summary.insert(0, "f(n)", ["g(n)", "h(n)", "g(n) + h(n)"])
summary["เหมาะที่สุด?"] = ["ใช่" if r.cost == ucs_result.cost else "ไม่" for r in best_first_results]
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, r in zip(axes, best_first_results):
    draw_factory_map(expanded=set(r.expanded), path=r.path, ax=ax,
                     title=f"{r.algorithm}: {r.cost} m, expanded {len(r.expanded)}")
plt.tight_layout()
plt.show()

**คำถามชวนคิด 3:** A* ไม่เคยขยาย A, B และ C เพราะค่า $f$ ของทั้งสาม (65, 75, 62) สูงกว่าต้นทุนคำตอบ 45 ถ้าวิศวกรย้ายสถานีประกอบ G ไปไว้ใกล้ C แทน คาดว่าจุดใดจะถูกขยายบ้าง ลองเปลี่ยนพิกัดของ G แล้วรันใหม่ (อย่าลืมว่าระยะทางวิ่งต้องไม่สั้นกว่าระยะเส้นตรง มิฉะนั้นฮิวริสติกจะประเมินเกินจริง)

## ส่วนที่ 4 ฮิวริสติกที่ยอมรับได้และคงเส้นคงวา

A* รับประกันคำตอบที่ดีที่สุดเมื่อฮิวริสติก

- **ยอมรับได้ (Admissible):** $h(n) \le h^*(n)$ ทุกโหนด เมื่อ $h^*(n)$ คือต้นทุนจริงที่ถูกที่สุดจาก $n$ ถึงเป้าหมาย
- **คงเส้นคงวา (Consistent):** $h(n) \le c(n, a, n') + h(n')$ ทุกเส้นเชื่อม (อสมการสามเหลี่ยม)

### 4.1 ตรวจการยอมรับได้ (ตัวอย่างที่ 2.16 ขั้นที่ 1)

หา $h^*(n)$ ด้วยการรัน UCS จากทุกจุดไปยัง G (ในผังเล็กทำได้ แต่ในงานจริงถ้ารู้ $h^*$ ก็ไม่ต้องค้นหาแล้ว)

In [ ]:
def true_costs_to_goal(graph, goal):
    """คืน dict {จุด: (h*, เส้นทางที่ดีที่สุด)} โดยรัน UCS จากแต่ละจุดไปยังเป้าหมาย"""
    answers = {}
    for node in graph:
        result = uniform_cost_search(RouteProblem(graph, node, goal))
        answers[node] = (result.cost, "-".join(result.path))
    return answers


h_star = true_costs_to_goal(FACTORY_MAP, GOAL)
pd.DataFrame([{"จุด": n, "เส้นทางที่ดีที่สุดถึง G": path, "h*(n)": cost, "h(n)": num(H_SLD[n]),
               "h ≤ h*?": "✓" if H_SLD[n] <= cost else "✗"}
              for n, (cost, path) in h_star.items()]).set_index("จุด")

(สำหรับจุด A มีเส้นทางที่ดีที่สุดสองเส้นยาว 65 m เท่ากัน คือ A-S-X-Y-G และ A-B-X-Y-G โค้ดแสดงเส้นที่ UCS พบก่อน)

### 4.2 ตรวจความคงเส้นคงวาทุกเส้นเชื่อม (ตัวอย่างที่ 2.16 ขั้นที่ 2)

หนังสือแสดงตัวอย่างเพียง 6 เส้นเชื่อมและให้ผู้อ่านตรวจที่เหลือเอง โค้ดด้านล่างตรวจครบทั้ง 18 ทิศ (9 เส้นเชื่อม × 2 ทิศ)

In [ ]:
def consistency_table(graph, h):
    """ตรวจ h(n) <= c(n, n') + h(n') ทุกเส้นเชื่อมทั้งสองทิศ คืนเป็นตาราง"""
    rows = []
    for n, nbrs in graph.items():
        for n2, cost in nbrs.items():
            rows.append({"เส้นเชื่อม n → n'": f"{n} → {n2}", "h(n)": num(h[n]),
                         "c + h(n')": f"{cost} + {num(h[n2])} = {num(cost + h[n2])}",
                         "ผ่าน?": h[n] <= cost + h[n2]})
    return pd.DataFrame(rows).set_index("เส้นเชื่อม n → n'")


consistency = consistency_table(FACTORY_MAP, H_SLD)
display(consistency)
print("จำนวนเส้นเชื่อม (นับทั้งสองทิศ) ที่ไม่ผ่าน:", (~consistency["ผ่าน?"]).sum())

### 4.3 ถ้าฮิวริสติกประเมินเกินจริง (ตัวอย่างที่ 2.16 ขั้นที่ 3)

สมมติวิศวกรตั้ง $h(Y) = 40$ เพราะเข้าใจผิดว่าโซน Y มีคนเดินพลุกพล่าน ค่านี้เกิน $h^*(Y) = 10$ จึงยอมรับไม่ได้ ไล่ A* ใหม่จะได้คำตอบ 60 m ซึ่งไม่ดีที่สุด

In [ ]:
H_BAD = dict(H_SLD, Y=40)            # สำเนาของ H_SLD ที่แก้เฉพาะ h(Y)
bad_trace = []
bad_result = astar_search(factory_problem, H_BAD.get, trace=bad_trace, name="A* (h(Y) = 40)")
show_trace(bad_trace)
print("เส้นทาง:", " → ".join(bad_result.path), "| ระยะทาง", bad_result.cost, "m")
print("h(Y) = 40 ยอมรับได้หรือไม่:", H_BAD["Y"] <= h_star["Y"][0])

**คำถามชวนคิด 4:** ถ้าต้องการให้ AGV หลีกเลี่ยงโซน Y ที่มีคนพลุกพล่านจริง ๆ ควรแก้ที่ใด ระหว่างฮิวริสติกกับต้นทุนของเส้นเชื่อม เพราะอะไร (จะได้ลองทำในแบบฝึกข้อ 3)

## ส่วนที่ 5 A* บนแผนที่ตารางของคลังสินค้า

### 5.1 นิยามปัญหาแผนที่ตาราง

คลังสินค้าแบ่งพื้นเป็นช่องตาราง `#` คือชั้นวางหรือผนัง `S` คือจุดเริ่มของ AGV และ `G` คือจุดรับสินค้า สถานะคือ `(แถว, คอลัมน์)` โดยแถว 0 อยู่บนสุด คลาส `GridProblem` รองรับการเคลื่อนที่สองแบบ

- **4 ทิศ** (ขึ้น ลง ซ้าย ขวา) ช่องละ 1 หน่วย
- **8 ทิศ** เพิ่มแนวทแยง ช่องละ $\sqrt{2} \approx 1.414$ หน่วย และห้าม "ตัดมุม" ชั้นวาง (จะวิ่งทแยงได้ต่อเมื่อช่องแนวตั้งและแนวนอนที่ติดกันว่างทั้งคู่) ตามตัวอย่างที่ 2.18

แผนที่นี้มีผนังยาวขวางระหว่าง S กับ G ผนังลักษณะนี้ทำให้ฮิวริสติก "ดึง" การค้นหาเข้าหาผนังได้ ซึ่งเป็นสถานการณ์ที่ Greedy หลงทางตามที่หนังสือกล่าวถึงในหัวข้อ 2.3.2

In [ ]:
WAREHOUSE = [
    "..........................",
    "......................G...",
    "..........................",
    "....##############........",
    ".................#........",
    "..######..######.#..####..",
    ".................#........",
    "..######..######.#..####..",
    ".................#........",
    "..S.......#......#........",
    "..........................",
]


class GridProblem:
    """แผนที่ตาราง: สถานะ = (แถว, คอลัมน์), '#' = ชั้นวาง, diagonal=True ให้วิ่งทแยงได้"""
    MOVES_4 = [("N", -1, 0), ("S", 1, 0), ("W", 0, -1), ("E", 0, 1)]
    MOVES_8 = MOVES_4 + [("NW", -1, -1), ("NE", -1, 1), ("SW", 1, -1), ("SE", 1, 1)]
    STEP = {name: (d_row, d_col) for name, d_row, d_col in MOVES_8}

    def __init__(self, grid, diagonal=False):
        self.grid = grid
        self.diagonal = diagonal
        self.n_rows, self.n_cols = len(grid), len(grid[0])
        for r, line in enumerate(grid):
            for c, ch in enumerate(line):
                if ch == "S":
                    self.initial = (r, c)
                elif ch == "G":
                    self.goal = (r, c)

    def is_free(self, r, c):
        return 0 <= r < self.n_rows and 0 <= c < self.n_cols and self.grid[r][c] != "#"

    def actions(self, state):
        r, c = state
        moves = self.MOVES_8 if self.diagonal else self.MOVES_4
        allowed = []
        for name, d_row, d_col in moves:
            if not self.is_free(r + d_row, c + d_col):
                continue
            if d_row and d_col and not (self.is_free(r + d_row, c) and self.is_free(r, c + d_col)):
                continue                      # ห้ามตัดมุมชั้นวาง
            allowed.append(name)
        return allowed

    def result(self, state, action):
        d_row, d_col = self.STEP[action]
        return (state[0] + d_row, state[1] + d_col)

    def action_cost(self, state, action, next_state):
        diagonal_move = state[0] != next_state[0] and state[1] != next_state[1]
        return math.sqrt(2) if diagonal_move else 1

    def is_goal(self, state):
        return state == self.goal


grid4 = GridProblem(WAREHOUSE)
print("เริ่มที่", grid4.initial, "เป้าหมาย", grid4.goal,
      "| ช่องที่วิ่งได้", sum(ch != "#" for line in WAREHOUSE for ch in line), "ช่อง")
draw_grid(WAREHOUSE, title="Warehouse grid map (# = rack / wall)")
plt.show()

### 5.2 AGV 4 ทิศ: BFS, UCS, Greedy และ A*

ฮิวริสติกบนแผนที่ตารางรับสถานะ `(แถว, คอลัมน์)` ฟังก์ชัน `make_heuristic` สร้างฮิวริสติกที่ "จำ" ตำแหน่งเป้าหมายไว้ จากฟังก์ชันระยะทางในส่วนที่ 1 สำหรับ AGV 4 ทิศ ระยะแมนฮัตตันคือคำตอบของปัญหาผ่อนคลาย "ไม่มีชั้นวางขวาง" จึงยอมรับได้ (หัวข้อ 2.3.4)

In [ ]:
def make_heuristic(distance, goal):
    """สร้างฮิวริสติก h(state) = distance(state, goal)"""
    return lambda state: distance(state, goal)


h_manhattan = make_heuristic(manhattan, grid4.goal)
h_euclid = make_heuristic(euclidean, grid4.goal)

grid_results = [
    breadth_first_search(grid4),
    uniform_cost_search(grid4),
    greedy_best_first_search(grid4, h_manhattan, name="Greedy (Manhattan)"),
    astar_search(grid4, h_manhattan, name="A* (Manhattan)"),
    astar_search(grid4, h_euclid, name="A* (straight-line)"),
]
display(pd.DataFrame([r.summary(show_path=False) for r in grid_results]).set_index("อัลกอริทึม"))

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for ax, r in zip(axes.flat, grid_results[:4]):
    draw_grid(WAREHOUSE, path=r.path, expanded=r.expanded, ax=ax,
              title=f"{r.algorithm}: cost {num(r.cost)}, expanded {len(r.expanded)}")
plt.tight_layout()
plt.show()

สิ่งที่ควรสังเกต

- **BFS และ UCS** ขยายช่องเกือบทั้งคลังแบบรอบทิศ เพราะไม่รู้ว่าเป้าหมายอยู่ทางใด (ทุกช่องมีต้นทุนเท่ากัน BFS จึงได้คำตอบที่ดีที่สุดเช่นเดียวกับ UCS)
- **Greedy** ขยายช่องน้อยที่สุด แต่พุ่งไปทางขวาเข้าหาผนังแนวตั้ง (คอลัมน์ 17) เพราะช่องเหล่านั้น "ดูใกล้" เป้าหมายตามฮิวริสติก เมื่อชนผนังจึงต้องวกลงไปอ้อมปลายผนังด้านล่าง ได้เส้นทาง 40 ช่อง ยาวกว่าคำตอบที่ดีที่สุด 30 ช่อง ขณะที่เส้นทางที่ดีที่สุดอ้อมผนังทางด้านบนซ้าย
- **A* (Manhattan)** ได้ต้นทุนเท่ากับ UCS โดยขยายช่องน้อยกว่ามาก ส่วน **A* (straight-line)** ก็ได้คำตอบที่ดีที่สุดเช่นกัน แต่ขยายมากกว่า เพราะระยะแมนฮัตตันครอบงำ (Dominate) ระยะเส้นตรงเมื่อเคลื่อนที่ 4 ทิศ ($h_{\text{MD}} \ge h_{\text{SLD}}$ ทุกช่อง)

### 5.3 AGV 8 ทิศ: เลือกฮิวริสติกให้ตรงกับการเคลื่อนที่ (ตัวอย่างที่ 2.18)

เมื่อ AGV วิ่งทแยงได้ ระยะแมนฮัตตันประเมินเกินจริง เช่น จาก (0, 0) ไป (1, 1) แมนฮัตตันบอก 2 แต่วิ่งทแยงจริงใช้เพียง 1.414 จึงยอมรับไม่ได้ ลองเทียบผลกับ UCS (ซึ่งรับประกันคำตอบที่ดีที่สุด)

In [ ]:
grid8 = GridProblem(WAREHOUSE, diagonal=True)
grid8_results = [
    uniform_cost_search(grid8),
    astar_search(grid8, h_manhattan, name="A* (Manhattan)"),
    astar_search(grid8, h_euclid, name="A* (straight-line)"),
]
optimal_cost_8 = grid8_results[0].cost
table8 = pd.DataFrame([r.summary(show_path=False) for r in grid8_results]).set_index("อัลกอริทึม")
table8["เหมาะที่สุด?"] = ["ใช่" if abs(r.cost - optimal_cost_8) < 1e-9 else "ไม่" for r in grid8_results]
display(table8)

fig, axes = plt.subplots(1, 3, figsize=(18, 3.8))
for ax, r in zip(axes, grid8_results):
    draw_grid(WAREHOUSE, path=r.path, expanded=r.expanded, ax=ax,
              title=f"8-dir {r.algorithm}: cost {num(r.cost)}, expanded {len(r.expanded)}")
plt.tight_layout()
plt.show()

A* ที่ใช้ระยะแมนฮัตตันขยายช่องน้อย แต่ได้เส้นทาง **ยาวกว่าคำตอบที่ดีที่สุด** เพราะฮิวริสติกที่ประเมินเกินจริงทำให้ A* มองข้ามเส้นทางทแยงที่ดีกว่า (เช่นเดียวกับ $h(Y) = 40$ ในส่วนที่ 4) ส่วนระยะเส้นตรงยังยอมรับได้ แต่ห่างจากต้นทุนจริงมาก จึงขยายช่องมาก แบบฝึกข้อ 1 จะให้เขียน **ระยะอ็อกไทล์ (Octile Distance)** ซึ่งยอมรับได้และใกล้ต้นทุนจริงกว่า

**คำถามชวนคิด 5:** ในตาราง 4 ทิศ A* (Manhattan) ขยายช่องน้อยกว่า Greedy หรือไม่ ถ้าไม่ แลกกับอะไร ในงาน AGV ที่วิ่งเส้นทางเดิมวันละหลายร้อยรอบ ควรเลือกอัลกอริทึมใด

## ส่วนที่ 6 8-Puzzle: เปรียบเทียบ $h_1$ กับ $h_2$

### 6.1 นิยามปัญหาและฮิวริสติก (ตัวอย่างที่ 2.2 และ 2.17)

สถานะของ 8-Puzzle เก็บเป็น tuple 9 ช่องเรียงจากแถวบนซ้ายไปแถวล่างขวา ใช้ 0 แทนช่องว่าง การกระทำคือเลื่อน **ช่องว่าง** ไปทาง ซ้าย ขวา ขึ้น หรือลง ฮิวริสติกสองตัวจากปัญหาผ่อนคลายในตัวอย่างที่ 2.17 คือ

- $h_1$ = จำนวนแผ่นที่วางผิดตำแหน่ง (ไม่นับช่องว่าง)
- $h_2$ = ผลรวมระยะแมนฮัตตันของแต่ละแผ่นถึงตำแหน่งเป้าหมาย

สถานะเริ่มต้นในรูปที่ 2.2 (7 2 4 / 5 _ 6 / 8 3 1) ควรได้ $h_1 = 8$ และ $h_2 = 18$

In [ ]:
GOAL_BOARD = (0, 1, 2, 3, 4, 5, 6, 7, 8)       # _ 1 2 / 3 4 5 / 6 7 8
START_BOARD = (7, 2, 4, 5, 0, 6, 8, 3, 1)      # 7 2 4 / 5 _ 6 / 8 3 1 (รูปที่ 2.2)
GOAL_POSITION = {tile: divmod(i, 3) for i, tile in enumerate(GOAL_BOARD)}


class PuzzleProblem:
    """8-Puzzle: การกระทำคือทิศที่ช่องว่าง (0) เคลื่อนไป ต้นทุนการเลื่อนละ 1"""
    MOVES = {"ซ้าย": (0, -1), "ขวา": (0, 1), "ขึ้น": (-1, 0), "ลง": (1, 0)}

    def __init__(self, initial, goal=GOAL_BOARD):
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        r, c = divmod(state.index(0), 3)
        return [a for a, (dr, dc) in self.MOVES.items() if 0 <= r + dr < 3 and 0 <= c + dc < 3]

    def result(self, state, action):
        blank = state.index(0)
        dr, dc = self.MOVES[action]
        target = blank + 3 * dr + dc
        board = list(state)
        board[blank], board[target] = board[target], board[blank]
        return tuple(board)

    def action_cost(self, state, action, next_state):
        return 1

    def is_goal(self, state):
        return state == self.goal


def board_text(state):
    return " / ".join(" ".join(str(t) if t else "_" for t in state[r:r + 3]) for r in (0, 3, 6))


def h1_misplaced(state):
    """h1: จำนวนแผ่นที่ไม่อยู่ที่ของตัวเอง (ไม่นับช่องว่าง)"""
    return sum(1 for i, tile in enumerate(state) if tile != 0 and tile != GOAL_BOARD[i])


def h2_manhattan(state):
    """h2: ผลรวมระยะแมนฮัตตันของทุกแผ่นถึงตำแหน่งเป้าหมาย"""
    total = 0
    for i, tile in enumerate(state):
        if tile != 0:
            row, col = divmod(i, 3)
            goal_row, goal_col = GOAL_POSITION[tile]
            total += abs(row - goal_row) + abs(col - goal_col)
    return total


puzzle = PuzzleProblem(START_BOARD)
print("สถานะเริ่มต้น:", board_text(START_BOARD), "| เป้าหมาย:", board_text(GOAL_BOARD))
for action in puzzle.actions(START_BOARD):
    print(f"  ช่องว่างไป{action}: {board_text(puzzle.result(START_BOARD, action))}")

# ตารางรายแผ่นเหมือนตัวอย่างที่ 2.17 ตำแหน่งเป็น (แถว, คอลัมน์) เริ่มจาก (0, 0) ที่มุมซ้ายบน
tile_rows = []
for tile in range(1, 9):
    position = divmod(START_BOARD.index(tile), 3)
    tile_rows.append({"แผ่น": tile, "ตำแหน่งปัจจุบัน": position, "ตำแหน่งเป้าหมาย": GOAL_POSITION[tile],
                      "วางผิด?": "✓" if position != GOAL_POSITION[tile] else "",
                      "ระยะแมนฮัตตัน": manhattan(position, GOAL_POSITION[tile])})
tile_rows.append({"แผ่น": "รวม", "ตำแหน่งปัจจุบัน": "", "ตำแหน่งเป้าหมาย": "",
                  "วางผิด?": f"h1 = {h1_misplaced(START_BOARD)}", "ระยะแมนฮัตตัน": f"h2 = {h2_manhattan(START_BOARD)}"})
pd.DataFrame(tile_rows).set_index("แผ่น")

### 6.2 แก้ปริศนาจากรูปที่ 2.2 ด้วย BFS, A* + $h_1$ และ A* + $h_2$

หนังสืออ้างจาก Russell และ Norvig (2021) ว่าคำตอบที่ดีที่สุดของสถานะนี้ใช้ 26 การเลื่อน ทั้งสามอัลกอริทึมควรได้ 26 เท่ากัน (BFS เหมาะที่สุดเพราะทุกการเลื่อนมีต้นทุนเท่ากัน) แต่จำนวนโหนดต่างกันมาก

**ตัวประกอบการแตกกิ่งยังผล** $b^*$ (หัวข้อ 2.3.4) คือค่าที่ทำให้ต้นไม้สม่ำเสมอลึก $d$ มีโหนด $N + 1$ โหนด เมื่อ $N$ คือจำนวนโหนดที่ถูกสร้าง

$$N + 1 = 1 + b^* + (b^*)^2 + \cdots + (b^*)^d$$

สมการนี้แก้ด้วยสูตรตรง ๆ ไม่ได้ จึงใช้ **วิธีแบ่งครึ่งช่วง (Bisection)** หา $b^*$ เริ่มจากตรวจกับตัวอย่างในหนังสือ ($N = 52$, $d = 5$ ได้ $b^* \approx 1.92$)

> **หมายเหตุ** เวลาที่วัดได้ขึ้นกับเครื่องที่รัน แต่จำนวนโหนดเท่าเดิมทุกครั้ง

In [ ]:
import time


def effective_branching_factor(n_generated, depth, tolerance=1e-6):
    """หา b* จาก N + 1 = 1 + b* + ... + (b*)^d ด้วยวิธีแบ่งครึ่งช่วง"""
    if depth == 0:
        return float("nan")
    low, high = 1.0, max(2.0, float(n_generated))
    while high - low > tolerance:
        mid = (low + high) / 2
        if sum(mid ** k for k in range(depth + 1)) < n_generated + 1:
            low = mid
        else:
            high = mid
    return (low + high) / 2


print("ตรวจกับหนังสือ: N = 52, d = 5 ได้ b* =", round(effective_branching_factor(52, 5), 2))


def run_timed(name, search):
    start = time.perf_counter()
    result = search()
    elapsed = time.perf_counter() - start
    n_generated = len(result.generated) - 1             # ไม่นับราก
    return {"อัลกอริทึม": name, "จำนวนการเลื่อน": result.cost,
            "โหนดที่ขยาย": len(result.expanded_nodes), "โหนดที่สร้าง N": n_generated,
            "b*": round(effective_branching_factor(n_generated, result.cost), 3),
            "เวลา (วินาที)": round(elapsed, 2)}


puzzle_rows = [
    run_timed("BFS", lambda: breadth_first_search(puzzle)),
    run_timed("A* + h1 (misplaced)", lambda: astar_search(puzzle, h1_misplaced)),
    run_timed("A* + h2 (Manhattan)", lambda: astar_search(puzzle, h2_manhattan)),
]
pd.DataFrame(puzzle_rows).set_index("อัลกอริทึม")

### 6.3 ทดลองกับปริศนาหลายข้อ

ผลจากปริศนาข้อเดียวอาจบังเอิญ เซลล์นี้สุ่มสร้างปริศนาด้วยการเลื่อนช่องว่างแบบสุ่มจากสถานะเป้าหมาย (ทุกข้อจึงแก้ได้แน่นอน) แล้วจัดกลุ่มตามความยาวของคำตอบที่ดีที่สุด $d$ กลุ่มละ 6 ข้อ จากนั้นหาค่าเฉลี่ยของจำนวนโหนดที่สร้างและ $b^*$ ของ A* ทั้งสองฮิวริสติก ลักษณะเดียวกับการทดลองใน Russell และ Norvig (2021) ที่หนังสืออ้างถึง

In [ ]:
def random_walk_board(n_moves, rng):
    """เลื่อนช่องว่างแบบสุ่มจากสถานะเป้าหมาย n_moves ครั้ง (ไม่เลื่อนย้อนกลับทันที)"""
    problem = PuzzleProblem(GOAL_BOARD)
    board, previous = GOAL_BOARD, None
    for _ in range(n_moves):
        options = [problem.result(board, a) for a in problem.actions(board)]
        options = [b for b in options if b != previous] or options
        previous, board = board, rng.choice(options)
    return board


rng = random.Random(RANDOM_SEED)
target_depths, per_depth = [4, 8, 12, 16, 20, 24], 6
samples = {d: [] for d in target_depths}
attempts = 0
while any(len(v) < per_depth for v in samples.values()) and attempts < 5000:
    attempts += 1
    board = random_walk_board(rng.randint(4, 120), rng)
    problem = PuzzleProblem(board)
    result_h2 = astar_search(problem, h2_manhattan)       # A* + h2 ให้ความยาวคำตอบที่ดีที่สุด d
    d = result_h2.cost
    if d in samples and len(samples[d]) < per_depth:
        result_h1 = astar_search(problem, h1_misplaced)
        samples[d].append((len(result_h1.generated) - 1, len(result_h2.generated) - 1))

experiment_rows = []
for d, pairs in samples.items():
    n_h1 = np.mean([p[0] for p in pairs])
    n_h2 = np.mean([p[1] for p in pairs])
    experiment_rows.append({"d": d, "จำนวนข้อ": len(pairs), "N เฉลี่ย (h1)": round(n_h1), "N เฉลี่ย (h2)": round(n_h2),
                            "b* (h1)": round(effective_branching_factor(n_h1, d), 2),
                            "b* (h2)": round(effective_branching_factor(n_h2, d), 2),
                            "h1 / h2": round(n_h1 / n_h2, 1)})
experiment = pd.DataFrame(experiment_rows).set_index("d")
print(f"สุ่มทั้งหมด {attempts} ข้อเพื่อให้ได้กลุ่มละ {per_depth} ข้อ")
display(experiment)

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.semilogy(experiment.index, experiment["N เฉลี่ย (h1)"], "o-", color=COLORS["orange"], label="A* + h1 (misplaced)")
ax.semilogy(experiment.index, experiment["N เฉลี่ย (h2)"], "s-", color=COLORS["blue"], label="A* + h2 (Manhattan)")
ax.set_xlabel("solution depth d")
ax.set_ylabel("nodes generated (log scale)")
ax.set_title("8-Puzzle: h2 dominates h1")
ax.legend()
plt.show()

เมื่อคำตอบลึกขึ้น จำนวนโหนดของ $h_1$ โตเร็วกว่าของ $h_2$ มาก และ $b^*$ ของ $h_2$ ต่ำกว่าเสมอ ผลนี้สอดคล้องกับหลัก **การครอบงำ** ในหัวข้อ 2.3.4 คือ $h_2(n) \ge h_1(n)$ ทุกสถานะ และทั้งคู่ยอมรับได้ A* ที่ใช้ $h_2$ จึงไม่ขยายโหนดมากกว่า (ที่ $d$ ตื้นมาก ทั้งสองแทบเท่ากัน เพราะ A* แทบเดินตรงไปถึงเป้าหมายอยู่แล้ว)

**คำถามชวนคิด 6:** ฮิวริสติก $h(n) = h_1(n) + h_2(n)$ มีค่ามากกว่าทั้งคู่ ทำไมจึงไม่ควรใช้ แล้ว $\max\{h_1(n), h_2(n)\}$ ต่างกันอย่างไร (ใบ้: ลองคำนวณ $h_1 + h_2$ ของสถานะเริ่มต้นเทียบกับต้นทุนจริง 26)

## ส่วนที่ 7 แบบฝึกปฏิบัติ

เติมโค้ดในตำแหน่งที่มี `# TODO` แล้วรันเซลล์ตรวจคำตอบที่อยู่ถัดไป ถ้ายังไม่ได้เติมโค้ด เซลล์ตรวจจะแจ้งว่า "ยังไม่ได้ทำ" โดยไม่หยุดการทำงานของ notebook

### ข้อ 1 ระยะอ็อกไทล์สำหรับ AGV 8 ทิศ

ตามตัวอย่างที่ 2.18 ถ้าไม่มีสิ่งกีดขวาง AGV ที่ต้องไปไกล $\Delta_r$ แถวและ $\Delta_c$ คอลัมน์ วิ่งทแยง $\min(\Delta_r, \Delta_c)$ ก้าว แล้ววิ่งตรงอีก $\max - \min$ ก้าว จึงได้

$$h_{\text{oct}} = \sqrt{2}\,\min(\Delta_r, \Delta_c) + \big(\max(\Delta_r, \Delta_c) - \min(\Delta_r, \Delta_c)\big)$$

จงเติมฟังก์ชัน `octile` (รับช่องสองช่อง คืนระยะอ็อกไทล์) แล้วเซลล์ตรวจจะใช้ `make_heuristic(octile, ...)` กับ A* บนแผนที่ 8 ทิศ

In [ ]:
def octile(cell_a, cell_b):
    """ระยะอ็อกไทล์ระหว่างช่อง (แถว, คอลัมน์) สองช่อง"""
    d_row = abs(cell_a[0] - cell_b[0])
    d_col = abs(cell_a[1] - cell_b[1])
    # TODO: คืนค่า sqrt(2) * min(d_row, d_col) + (max(d_row, d_col) - min(d_row, d_col))
    return 0

In [ ]:
# เซลล์ตรวจคำตอบข้อ 1 (ไม่ต้องแก้)
if octile((1, 1), (4, 3)) == 0:
    print("ข้อ 1: ยังไม่ได้ทำ")
elif abs(octile((1, 1), (4, 3)) - 3.83) > 0.005:
    print("ข้อ 1: ยังไม่ถูก — ตัวอย่างที่ 2.18 จาก (1,1) ไป (4,3) ต้องได้ 3.83 แต่ได้", round(octile((1, 1), (4, 3)), 3))
else:
    h_octile = make_heuristic(octile, grid8.goal)
    octile_result = astar_search(grid8, h_octile, name="A* (octile)")
    all8 = grid8_results + [octile_result]
    table = pd.DataFrame([r.summary(show_path=False) for r in all8]).set_index("อัลกอริทึม")
    table["เหมาะที่สุด?"] = ["ใช่" if abs(r.cost - optimal_cost_8) < 1e-9 else "ไม่" for r in all8]
    display(table)
    optimal = abs(octile_result.cost - optimal_cost_8) < 1e-9
    fewer = len(octile_result.expanded) < len(grid8_results[2].expanded)
    print("ข้อ 1:", "ถูกต้อง —" if optimal and fewer else "ตรวจอีกครั้ง —",
          f"อ็อกไทล์ได้ต้นทุน {num(octile_result.cost)} (ดีที่สุด = {num(optimal_cost_8)})",
          f"ขยาย {len(octile_result.expanded)} ช่อง เทียบกับระยะเส้นตรง {len(grid8_results[2].expanded)} ช่อง")
    draw_grid(WAREHOUSE, path=octile_result.path, expanded=octile_result.expanded,
              title=f"8-dir A* (octile): cost {num(octile_result.cost)}, expanded {len(octile_result.expanded)}")
    plt.show()

**คำถามข้อ 1:** ทำไมระยะอ็อกไทล์จึงยอมรับได้สำหรับ AGV 8 ทิศ และทำไมจึงขยายช่องน้อยกว่าระยะเส้นตรง ทั้งที่ทั้งสองยอมรับได้

*คำตอบ:* (เขียนคำตอบที่นี่)

### ข้อ 2 A* แบบถ่วงน้ำหนัก (Weighted A*)

ในงานที่ต้องการคำตอบเร็ว บางระบบใช้ $f(n) = g(n) + w \cdot h(n)$ โดย $w > 1$ ซึ่งอยู่ระหว่าง A* ($w = 1$) กับ Greedy ($w \to \infty$) จงเติมฟังก์ชัน `weighted_astar_search` โดยเรียก `best_first_search` (ดู `astar_search` ในส่วนที่ 2 เป็นแบบอย่าง) แล้วเซลล์ตรวจจะรันบนแผนที่ 4 ทิศด้วยระยะแมนฮัตตันที่ $w$ = 1, 1.5, 2, 3 และ 5

In [ ]:
def weighted_astar_search(problem, h, weight):
    """Weighted A*: f(n) = g(n) + weight * h(n)"""
    # TODO: เรียก best_first_search(problem, f=..., name=f"Weighted A* (w = {weight})",
    #       describe=lambda n: str(n.state)) แล้วคืนผลลัพธ์
    return None

In [ ]:
# เซลล์ตรวจคำตอบข้อ 2 (ไม่ต้องแก้)
if weighted_astar_search(grid4, h_manhattan, 1) is None:
    print("ข้อ 2: ยังไม่ได้ทำ")
else:
    weights = [1, 1.5, 2, 3, 5]
    weighted = [weighted_astar_search(grid4, h_manhattan, w) for w in weights]
    optimal_cost_4 = grid_results[1].cost                   # ต้นทุนที่ดีที่สุดจาก UCS
    rows = [{"w": w, "ต้นทุน": num(r.cost), "ขอบบน w × C*": num(w * optimal_cost_4),
             "จำนวนช่องที่ขยาย": len(r.expanded)} for w, r in zip(weights, weighted)]
    display(pd.DataFrame(rows).set_index("w"))
    same_as_astar = weighted[0].cost == grid_results[3].cost and len(weighted[0].expanded) == len(grid_results[3].expanded)
    within_bound = all(r.cost <= w * optimal_cost_4 + 1e-9 for w, r in zip(weights, weighted))
    print("ข้อ 2:", "ถูกต้อง — w = 1 ให้ผลเท่ากับ A* และทุกค่า w ได้ต้นทุนไม่เกิน w × C*" if same_as_astar and within_bound
          else "ตรวจอีกครั้ง — w = 1 ต้องให้ผลเท่ากับ astar_search")

    fig, ax1 = plt.subplots(figsize=(6, 3.5))
    ax1.plot(weights, [len(r.expanded) for r in weighted], "o-", color=COLORS["blue"])
    ax1.set_xlabel("weight w")
    ax1.set_ylabel("cells expanded", color=COLORS["blue"])
    ax2 = ax1.twinx()
    ax2.plot(weights, [r.cost for r in weighted], "s--", color=COLORS["orange"])
    ax2.set_ylabel("path cost", color=COLORS["orange"])
    ax1.set_title("Weighted A*: speed vs path quality")
    fig.tight_layout()
    plt.show()

**คำถามข้อ 2:** เมื่อเพิ่ม $w$ จำนวนช่องที่ขยายและต้นทุนเส้นทางเปลี่ยนไปอย่างไร ถ้าระบบจัดการ AGV ต้องวางแผนเส้นทางใหม่ทุก 50 มิลลิวินาทีเมื่อมีคนเดินตัดหน้า ควรเลือก $w$ แบบใด เพราะอะไร

*คำตอบ:* (เขียนคำตอบที่นี่)

### ข้อ 3 สะท้อนความแออัดในต้นทุน ไม่ใช่ในฮิวริสติก

ต่อจากตัวอย่างที่ 2.16 ขั้นที่ 3 สมมติโซนระหว่าง X กับ Y มีคนเดินพลุกพล่านจริง AGV ต้องชะลอความเร็ว ทีมวิศวกรจึงคิดต้นทุนทางวิ่ง X–Y เพิ่มอีก 20 หน่วย (จาก 15 เป็น 35) จงสร้าง `busy_map` เป็น **สำเนา** ของ `FACTORY_MAP` ที่ทางวิ่ง X–Y มีต้นทุน 35 ทั้งสองทิศ โดยไม่แก้ `FACTORY_MAP` เดิม แล้วเซลล์ตรวจจะรัน A* ด้วยฮิวริสติกระยะเส้นตรงตัวเดิม

In [ ]:
# TODO: สร้าง busy_map เป็นสำเนาสองชั้นของ FACTORY_MAP (ดูวิธีใน close_edge ของปฏิบัติการที่ 3)
#       แล้วกำหนดต้นทุน X -> Y และ Y -> X เป็น 35
busy_map = None

In [ ]:
# เซลล์ตรวจคำตอบข้อ 3 (ไม่ต้องแก้)
if busy_map is None:
    print("ข้อ 3: ยังไม่ได้ทำ")
elif FACTORY_MAP["X"]["Y"] != 15:
    print("ข้อ 3: ระวัง! FACTORY_MAP เดิมถูกแก้ไป ให้สร้างสำเนาก่อนแก้ แล้วรัน notebook ใหม่ทั้งหมด")
elif busy_map["X"]["Y"] != 35 or busy_map["Y"]["X"] != 35:
    print("ข้อ 3: ยังไม่ถูก — ต้นทุน X–Y ต้องเป็น 35 ทั้งสองทิศ")
else:
    busy_problem = RouteProblem(busy_map, START, GOAL)
    busy_trace = []
    busy_astar = astar_search(busy_problem, h_sld, trace=busy_trace, name="A* (busy X-Y)")
    show_trace(busy_trace)
    busy_ucs = uniform_cost_search(busy_problem)
    violations = (~consistency_table(busy_map, H_SLD)["ผ่าน?"]).sum()
    print("A* :", " → ".join(busy_astar.path), f"({busy_astar.cost} m) ขยาย {len(busy_astar.expanded)} โหนด")
    print("UCS:", " → ".join(busy_ucs.path), f"({busy_ucs.cost} m) ขยาย {len(busy_ucs.expanded)} โหนด")
    print("จำนวนเส้นเชื่อมที่ละเมิดความคงเส้นคงวา:", violations)
    if busy_astar.path == ["S", "X", "G"] and busy_astar.cost == busy_ucs.cost == 60 and violations == 0:
        print("ข้อ 3: ถูกต้อง — A* เลี่ยงโซนแออัดและยังได้คำตอบที่ดีที่สุดเท่ากับ UCS")
    draw_factory_map(graph=busy_map, expanded=set(busy_astar.expanded), path=busy_astar.path,
                     title=f"Busy X-Y (cost 35): A* {busy_astar.cost}, expanded {len(busy_astar.expanded)}")
    plt.show()

**คำถามข้อ 3:** เมื่อเพิ่มต้นทุนของ X–Y ฮิวริสติกระยะเส้นตรงตัวเดิมยังยอมรับได้หรือไม่ เพราะอะไร เปรียบเทียบผลกับการตั้ง $h(Y) = 40$ ในส่วนที่ 4 แล้ววิธีใดถูกต้อง

*คำตอบ:* (เขียนคำตอบที่นี่)

## สรุป

- **ฮิวริสติก** $h(n)$ ประมาณต้นทุนที่เหลือถึงเป้าหมายโดยไม่ต้องค้นหา ระยะเส้นตรงของผังโรงงานตรงกับตารางที่ 2.8 และทั้งยอมรับได้และคงเส้นคงวาครบทั้ง 18 ทิศของเส้นเชื่อม (ส่วนที่ 1 และ 4)
- **Greedy** ($f = h$) ขยายเพียง 2 โหนดแต่ได้ 60 m ส่วน **A*** ($f = g + h$) ได้ 45 m เท่ากับ UCS โดยขยายเพียง 3 โหนดเทียบกับ 6 โหนดของ UCS ตรงกับตารางที่ 2.9 ทั้งสามอัลกอริทึมใช้ `best_first_search` ตัวเดียวกัน ต่างกันเพียง $f$ (ส่วนที่ 2–3)
- **ฮิวริสติกที่ประเมินเกินจริง** ทำให้ A* พลาดคำตอบที่ดีที่สุด ทั้งในผังโรงงาน ($h(Y) = 40$) และแผนที่ 8 ทิศ (ระยะแมนฮัตตัน) ความรู้เรื่องความแออัดต้องใส่ในต้นทุนเส้นเชื่อม ไม่ใช่ในฮิวริสติก (ส่วนที่ 4–5 และแบบฝึกข้อ 3)
- **บนแผนที่ตาราง** A* ขยายช่องน้อยกว่า BFS/UCS มากโดยได้คำตอบที่ดีที่สุดเท่ากัน และฮิวริสติกต้องตรงกับวิธีเคลื่อนที่: แมนฮัตตันสำหรับ 4 ทิศ อ็อกไทล์สำหรับ 8 ทิศ (ส่วนที่ 5 และแบบฝึกข้อ 1)
- **ฮิวริสติกที่ครอบงำดีกว่า**: ใน 8-Puzzle ทั้ง $h_1$ และ $h_2$ ได้คำตอบ 26 การเลื่อน แต่ $h_2$ สร้างโหนดน้อยกว่าหลายเท่า และมีตัวประกอบการแตกกิ่งยังผลต่ำกว่าเสมอ (ส่วนที่ 6)

**ก้าวต่อไป:** ปฏิบัติการที่ 5 เปลี่ยนจากการหา "เส้นทาง" ไปเป็นการหา "สถานะที่ดีที่สุด" ด้วย Hill Climbing, Simulated Annealing และ Genetic Algorithm (หัวข้อ 2.4)

## เอกสารอ้างอิง

- Russell, S., & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. บทที่ 3.5–3.6
- Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A formal basis for the heuristic determination of minimum cost paths. *IEEE Transactions on Systems Science and Cybernetics, 4*(2), 100–107.
- Pohl, I. (1970). Heuristic search viewed as path finding in a graph. *Artificial Intelligence, 1*(3–4), 193–204. (ต้นแบบของ Weighted A*)